# Train on the platform

The other training template fits a model **in this kernel**, which is the right
tool while you are still deciding what to fit. This one starts the platform's
own job: it appears on `/training`, streams the same logs, registers a model the
catalog knows about, and keeps running after you close the tab.

The reason to do it from a notebook rather than from the form is everything
*before* the call — picking the dataset by a rule, sweeping a parameter, or
kicking off four runs and going to lunch.

In [ ]:
import orinth

compute = orinth.train.compute()
for framework in compute["frameworks"]:
    devices = ", ".join(device["name"] for device in framework["devices"]) or "cpu only"
    print(f"{framework['name']:12} accelerated={str(framework['accelerated']):5}  {devices}")

## Pick the dataset and the model option

A **model option** is not a model: it is an entry in the training catalog
(`keras_mobilenet_v2`, `yolo_local`, …) that decides which runner executes and
what its defaults are. `options()` carries those defaults, so a notebook can
start from them instead of inventing numbers.

In [ ]:
TASK = "classification"

dataset = max(
    (
        candidate
        for candidate in orinth.datasets.list()
        if candidate.task_type == TASK and candidate.readiness.trainable
    ),
    key=lambda candidate: candidate.total_items,
)
option = next(entry for entry in orinth.train.options(TASK) if entry["runnable"])

print("dataset:", dataset.id, dataset.splits)
print("option :", option["id"], "|", option["description"][:90])
print("defaults:", option["defaults"])

## Start it

`wait=False` is the default and it is the point: training is minutes to hours,
and blocking the cell costs you the kernel for the duration. Start it, keep
working, and wait when you actually need the model.

Epochs are low here so the template finishes while you watch. Raise them.

In [ ]:
run = orinth.train.start(
    dataset.id,
    model_option_id=option["id"],
    task_type=TASK,
    name="from a notebook",
    epochs=3,
    image_size=160,
    batch_size=8,
)
print(run.id, run.status, "-> watch it on /training")

## Watch it

`wait_for` polls and calls `on_progress` with each read. A **failed** run comes
back as a value rather than an exception — it is a real outcome with an error
attached, and a cell that raised would lose the runs beside it.

In [ ]:
final = orinth.train.wait_for(
    run.id,
    on_progress=lambda current: print(f"{current.percent:5.1f}%  {current.step}"),
)

print()
print("status :", final.status)
print("model  :", final.model_id)
print("error  :", final.error)

## Use what it registered

`model_id` is the join to everything else: `models.predictor()`,
`evaluate.start()`, `inference.predict()` all take it. That is the whole reason
to train through the platform instead of in the kernel — the artifact is
addressable afterwards.

In [ ]:
if final.model_id:
    model = orinth.models.get(final.model_id)
    print(model)
    print("\nmetrics:")
    for key, value in sorted(final.metrics.items()):
        if isinstance(value, (int, float)):
            print(f"  {key:24} {value}")
else:
    print("Nothing registered — read final.error above.")